# GIK-IceChain v2.0 — End-to-End Pipeline Test
**ECMWF Code for Earth 2026 — Challenge 41: Zero-Cost Flood Risk for East Africa**

This notebook runs the full C1 → C2 → C3 pipeline and displays key metrics at each stage.

| Stage | Description | Data source |
|-------|-------------|-------------|
| **C1** | ECMWF IFS → IceChunk virtual store | HuggingFace `E4DRR/gik-ecmwf-par` |
| **C2** | Exceedance probabilities (adaptive GEV) | IceChunk store + synthetic thresholds |
| **C3** | CRMA Bayesian Network flood risk | Exceedance store + synthetic GPM |


## 0. Install

In [ ]:
%%capture cap
!pip install -q git+https://github.com/hashirama21/gik-icechain.git
!pip install -q \
    virtualizarr icechunk obstore obspec-utils \
    pyarrow fsspec huggingface_hub upath \
    matplotlib seaborn rich

print('Install done.')

In [ ]:
import time, warnings
import numpy as np
import pandas as pd
import xarray as xr
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from datetime import date, timedelta
from rich.console import Console
from rich.table import Table
from rich import print as rprint

warnings.filterwarnings('ignore')
console = Console()

# Timing helper
class Timer:
    def __init__(self, label):
        self.label = label
    def __enter__(self):
        self._t = time.perf_counter()
        return self
    def __exit__(self, *_):
        self.elapsed = time.perf_counter() - self._t
        rprint(f'[bold green]✓[/] {self.label}: [cyan]{self.elapsed:.2f}s[/]')

METRICS = {}  # collect all metrics for final dashboard
rprint('[bold]Environment ready.[/]')

---
## C1 — Ingest: ECMWF IFS → IceChunk Virtual Store

### C1.1 Load the HuggingFace GIK catalog

In [ ]:
from gik_icechain.conversion.gik_loader import GIKCatalog

with Timer('Catalog load (HuggingFace)'):
    catalog = GIKCatalog()
    df_cat = catalog.load_catalog()

n_dates   = df_cat['date'].nunique()
n_members = df_cat['member'].nunique()
n_files   = len(df_cat)
date_min  = df_cat['date'].min()
date_max  = df_cat['date'].max()
total_gb  = df_cat['size_bytes'].sum() / 1e9

METRICS['C1_catalog_rows']  = n_files
METRICS['C1_dates_covered'] = n_dates
METRICS['C1_members']       = n_members
METRICS['C1_catalog_gb']    = total_gb

t = Table(title='GIK Catalog Stats', show_header=True, header_style='bold cyan')
t.add_column('Metric');  t.add_column('Value', justify='right')
t.add_row('Total files',      f'{n_files:,}')
t.add_row('Dates covered',    f'{n_dates:,}')
t.add_row('Ensemble members', f'{n_members}')
t.add_row('Date range',       f'{date_min} → {date_max}')
t.add_row('Total metadata',   f'{total_gb:.1f} GB')
console.print(t)

### C1.2 Resolve Parquet paths for a test date

In [ ]:
TEST_DATE     = date(2024, 3, 1)
TEST_RUN_HOUR = 0
FLOOD_VARS    = ['tp', '2t', '10u', '10v', 'sro', 'ssro']

with Timer('Parquet path resolution'):
    paths = catalog.get_parquet_paths(
        start=TEST_DATE, end=TEST_DATE,
        run_hours=(TEST_RUN_HOUR,),
    )

control_paths = [p for p in paths if 'control' in p]
member_paths  = [p for p in paths if 'ens_' in p]

METRICS['C1_paths_resolved'] = len(paths)
rprint(f'[bold]{len(paths)}[/] files — [green]{len(control_paths)}[/] control + [green]{len(member_paths)}[/] perturbed members')
rprint(f'Example: [dim]{paths[0]}[/]')

### C1.3 Inspect raw Kerchunk references

In [ ]:
import fsspec

with Timer('Fetch control parquet'):
    with fsspec.open(control_paths[0], 'rb') as f:
        ref_df = pd.read_parquet(f)

# Count reference types
meta_keys  = ref_df[ref_df['key'].str.contains(r'\.z(array|attrs|group|metadata)', na=False)]
chunk_keys = ref_df[~ref_df['key'].str.contains(r'\.z', na=False)]

METRICS['C1_ref_rows']   = len(ref_df)
METRICS['C1_meta_refs']  = len(meta_keys)
METRICS['C1_chunk_refs'] = len(chunk_keys)

t = Table(title=f'Kerchunk Parquet: {control_paths[0].split("/")[-1]}')
t.add_column('Type'); t.add_column('Count', justify='right')
t.add_row('Total rows',    f'{len(ref_df):,}')
t.add_row('Metadata refs', f'{len(meta_keys):,}')
t.add_row('Chunk refs',    f'{len(chunk_keys):,}')
t.add_row('File size',     f'{ref_df.memory_usage(deep=True).sum()/1024:.0f} KB (in-memory)')
console.print(t)
ref_df.head(3)

### C1.4 Virtualize the control member (VirtualiZarr 2.x)

In [ ]:
from gik_icechain.conversion.virtualizer import parquet_to_virtual_dataset

with Timer('Virtualize control member'):
    vds = parquet_to_virtual_dataset([control_paths[0]], variables=FLOOD_VARS)

METRICS['C1_virtual_vars'] = len(vds.data_vars)
METRICS['C1_virtual_dims'] = dict(vds.dims)

rprint('[bold green]✓ Virtual dataset created (no data downloaded)[/]')
print(vds)

### C1.5 Virtualize a 3-member ensemble

In [ ]:
SAMPLE_SIZE = 3  # Use 3 members for speed; increase to 51 for full ensemble

sample_paths = control_paths + member_paths[:SAMPLE_SIZE - 1]

with Timer(f'Virtualize {SAMPLE_SIZE}-member ensemble'):
    vds_ens = parquet_to_virtual_dataset(sample_paths, variables=FLOOD_VARS)

rprint(f'[bold]Ensemble virtual dataset[/] ({SAMPLE_SIZE} members):')
print(vds_ens)

### C1.6 Coverage gap analysis

In [ ]:
with Timer('Coverage gap analysis (2024-Q1 → 2024-Q2)'):
    available_dates = catalog.list_available_dates()
    gaps = catalog.get_coverage_gap(
        start=date(2024, 3, 1),
        end=date(2024, 6, 30),
    )

n_expected = (date(2024, 6, 30) - date(2024, 3, 1)).days + 1
n_available = n_expected - len(gaps)
pct_coverage = 100 * n_available / n_expected

METRICS['C1_gap_days'] = len(gaps)
METRICS['C1_pct_coverage'] = pct_coverage

t = Table(title='Coverage 2024-03-01 → 2024-06-30')
t.add_column('Metric'); t.add_column('Value', justify='right')
t.add_row('Expected days',  str(n_expected))
t.add_row('Available days', str(n_available))
t.add_row('Missing days',   str(len(gaps)))
t.add_row('Coverage %',     f'{pct_coverage:.1f}%')
if gaps:
    t.add_row('First gap',  str(gaps[0]))
    t.add_row('Last gap',   str(gaps[-1]))
console.print(t)

# Visualise coverage
all_dates = pd.date_range(date(2024, 3, 1), date(2024, 6, 30))
gap_set   = set(str(g) for g in gaps)
coverage  = [0 if str(d.date()) in gap_set else 1 for d in all_dates]

fig, ax = plt.subplots(figsize=(14, 1.8))
ax.bar(all_dates, coverage, width=1, color=['#2ecc71' if c else '#e74c3c' for c in coverage])
ax.set_yticks([])
ax.set_xlabel('Date')
ax.set_title('GIK Dataset Coverage (green = available, red = gap)')
ax.xaxis.set_major_locator(plt.matplotlib.dates.MonthLocator())
ax.xaxis.set_major_formatter(plt.matplotlib.dates.DateFormatter('%b %Y'))
plt.tight_layout()
plt.show()

---
## C2 — Exceedance Probabilities

C2 needs actual ECMWF data from S3. We simulate the intermediate dataset with synthetic data that matches the expected schema, then run the full exceedance computation code.

### C2.1 Build a synthetic IceChunk-like dataset

In [ ]:
# Synthetic ECMWF-like dataset (matches what C1 would write to IceChunk)
# East Africa domain: 22°E–52°E, -12°N–22°N
np.random.seed(42)

LAT  = np.linspace(22, -12, 35)   # 1° resolution
LON  = np.linspace(22, 52, 31)
TIME = pd.date_range(TEST_DATE, periods=10, freq='6h')
STEP = np.arange(0, 120+1, 6)     # 0–120h in 6h steps
MEMBER = np.arange(0, 10)         # 0=control, 1–9=perturbed

# Total precipitation (mm) — random with East African seasonality
tp_data = np.random.exponential(scale=5, size=(len(TIME), len(STEP), len(MEMBER), len(LAT), len(LON)))
# Add spatial pattern: more rain near equator
lat_weight = np.exp(-((LAT[:, np.newaxis] - 0) ** 2) / 100)
tp_data *= lat_weight[np.newaxis, np.newaxis, np.newaxis, :, :]

ds_ecmwf = xr.Dataset(
    {'tp': (['time', 'step', 'member', 'latitude', 'longitude'], tp_data, {'units': 'mm', 'long_name': 'Total precipitation'})},
    coords={'time': TIME, 'step': STEP, 'member': MEMBER, 'latitude': LAT, 'longitude': LON},
)
rprint(f'[bold]Synthetic ECMWF dataset[/]: {dict(ds_ecmwf.dims)}')
print(ds_ecmwf)

### C2.2 Rolling precipitation accumulations

In [ ]:
from gik_icechain.exceedance.accumulations import compute_rolling_accumulations

WINDOWS_H = [24, 72, 120]  # 1-day, 3-day, 5-day

with Timer('Rolling accumulations'):
    acc_ds = compute_rolling_accumulations(ds_ecmwf, windows_h=WINDOWS_H)

METRICS['C2_windows']       = WINDOWS_H
METRICS['C2_acc_vars']      = list(acc_ds.data_vars)

rprint(f'[bold]Accumulation variables[/]: {list(acc_ds.data_vars)}')

# Plot ensemble spread for 24h accumulation over East Africa
fig, axes = plt.subplots(1, len(WINDOWS_H), figsize=(15, 4))
for ax, w in zip(axes, WINDOWS_H):
    var = f'tp_{w}h'
    if var in acc_ds:
        data = acc_ds[var].isel(time=0).mean('member').values
        im = ax.pcolormesh(LON, LAT, data, cmap='Blues', vmin=0)
        plt.colorbar(im, ax=ax, label='mm')
    ax.set_title(f'{w}h accumulation (ensemble mean)')
    ax.set_xlabel('Longitude'); ax.set_ylabel('Latitude')
plt.suptitle(f'Precipitation Accumulations — {TEST_DATE} {TEST_RUN_HOUR:02d}z')
plt.tight_layout()
plt.show()

### C2.3 Adaptive GEV thresholds

In [ ]:
from gik_icechain.exceedance.thresholds import (
    AdaptiveGEVThresholds, ClimateMode, ENSOPhase, IODPhase,
    get_season, classify_enso, classify_iod,
)

# Build synthetic GEV thresholds (normally loaded from CMORPH)
thr = AdaptiveGEVThresholds()
for window in WINDOWS_H:
    for rp in [2, 5, 10, 20]:
        for season in ['MAM', 'OND', 'DJF', 'JJA']:
            for enso in ENSOPhase:
                for iod in IODPhase:
                    mode = ClimateMode(season=season, enso=enso, iod=iod)
                    # Synthetic GEV threshold (gamma × return period)
                    base = np.random.gamma(2, scale=window * 0.5, size=(len(LAT), len(LON)))
                    thr_da = xr.DataArray(base * (1 + 0.3 * np.log(rp)),
                                         dims=['latitude', 'longitude'],
                                         coords={'latitude': LAT, 'longitude': LON},
                                         attrs={'units': 'mm', 'return_period_y': rp, 'window_h': window})
                    thr.set(window, rp, mode, thr_da)

season = get_season(TEST_DATE.month)
mode   = ClimateMode(season=season, enso=ENSOPhase.NEUTRAL, iod=IODPhase.NEUTRAL)
rprint(f'[bold]Climate mode for {TEST_DATE}[/]: {mode}')

sample_thr = thr.get(24, 5, mode)
fig, ax = plt.subplots(figsize=(7, 4))
im = ax.pcolormesh(LON, LAT, sample_thr.values, cmap='YlOrRd')
plt.colorbar(im, ax=ax, label='mm')
ax.set_title(f'5-year return period threshold — 24h, {season} / ENSO-neutral')
ax.set_xlabel('Longitude'); ax.set_ylabel('Latitude')
plt.tight_layout()
plt.show()

### C2.4 Compute exceedance probabilities

In [ ]:
from gik_icechain.exceedance.exceedance import compute_exceedance_probabilities

RETURN_PERIODS = [2, 5, 10, 20]
exc_results = {}

with Timer('Exceedance probabilities (all windows × return periods)'):
    for window in WINDOWS_H:
        for rp in RETURN_PERIODS:
            thr_ds = xr.Dataset({f'rp_{rp}y': thr.get(window, rp, mode)})
            acc_slice = acc_ds.isel(time=0)
            try:
                exc = compute_exceedance_probabilities(acc_slice, thr_ds, window_h=window, return_period=rp)
                exc_results[(window, rp)] = exc
            except Exception as e:
                rprint(f'[yellow]  skip ({window}h, {rp}y): {e}[/]')

n_computed = len(exc_results)
METRICS['C2_exceedance_maps'] = n_computed
rprint(f'[bold green]✓[/] {n_computed}/{len(WINDOWS_H)*len(RETURN_PERIODS)} exceedance maps computed')

# Plot exceedance probability for different return periods (24h window)
fig, axes = plt.subplots(1, len(RETURN_PERIODS), figsize=(18, 4))
for ax, rp in zip(axes, RETURN_PERIODS):
    key = (24, rp)
    if key in exc_results:
        data = exc_results[key]
        vals = data.values if isinstance(data, xr.DataArray) else np.zeros((len(LAT), len(LON)))
        im = ax.pcolormesh(LON, LAT, vals, cmap='RdYlGn_r', vmin=0, vmax=1)
        plt.colorbar(im, ax=ax, label='P(exceed)')
    ax.set_title(f'P(exceed {rp}y RP) — 24h')
    ax.set_xlabel('Longitude'); ax.set_ylabel('Latitude')
plt.suptitle(f'Exceedance Probabilities — {TEST_DATE} {TEST_RUN_HOUR:02d}z')
plt.tight_layout()
plt.show()

---
## C3 — Flood Risk (CRMA Bayesian Network)

### C3.1 Build the CRMA model

In [ ]:
from gik_icechain.risk.crma_model import CRMAModel, RiskEvidence

with Timer('CRMA model build (DiscreteBN + DBN)'):
    model = CRMAModel()
    model.build()

rprint('[bold green]✓ CRMA Bayesian Network built[/]')
rprint('[dim]Nodes: Forecast_Hazard → Obs_Antecedent → Temporal_Persist → Spatial_Coverage → Data_Confidence → API_State → Exposure → Risk_State[/]')

### C3.2 Single-step inference — risk scenarios

In [ ]:
scenarios = {
    'Low risk (dry season, no rain)': RiskEvidence(
        forecast_hazard=0, obs_antecedent=0, temporal_persist=0,
        spatial_coverage=0, data_confidence=2),
    'Moderate risk (wet season)': RiskEvidence(
        forecast_hazard=1, obs_antecedent=1, temporal_persist=1,
        spatial_coverage=1, data_confidence=1),
    'High risk (extreme event)': RiskEvidence(
        forecast_hazard=2, obs_antecedent=2, temporal_persist=1,
        spatial_coverage=2, data_confidence=1),
    'High risk + saturated soils': RiskEvidence(
        forecast_hazard=2, obs_antecedent=2, temporal_persist=1,
        spatial_coverage=2, data_confidence=0),
}

t = Table(title='CRMA Risk Inference — Scenario Analysis', show_header=True, header_style='bold magenta')
t.add_column('Scenario')
t.add_column('P(Low)', justify='right')
t.add_column('P(Medium)', justify='right')
t.add_column('P(High)', justify='right')
t.add_column('Dominant', justify='center')

risk_labels = ['Low', 'Medium', 'High']
all_probs   = []

for name, ev in scenarios.items():
    for api_state in [0, 2]:  # Dry / Saturated
        res = model.infer(ev, api_state=api_state)
        probs = res.get('probabilities', [0.33, 0.33, 0.34])
        dominant = risk_labels[int(np.argmax(probs))]
        row_label = f"{name}\n  (API={'Dry' if api_state==0 else 'Saturated'})"
        all_probs.append({'scenario': name, 'api': api_state, 'probs': probs})
        color = 'green' if dominant == 'Low' else ('yellow' if dominant == 'Medium' else 'red')
        t.add_row(
            row_label,
            f'{probs[0]:.3f}', f'{probs[1]:.3f}', f'{probs[2]:.3f}',
            f'[{color}]{dominant}[/{color}]'
        )

console.print(t)

METRICS['C3_scenarios'] = len(scenarios)

### C3.3 Temporal DBN inference — 7-day sequence

In [ ]:
# Simulate worsening conditions over 7 days (e.g. OND 2024 flash-flood event)
sequence = [
    RiskEvidence(forecast_hazard=0, obs_antecedent=0, temporal_persist=0, spatial_coverage=0, data_confidence=1),
    RiskEvidence(forecast_hazard=0, obs_antecedent=0, temporal_persist=0, spatial_coverage=1, data_confidence=1),
    RiskEvidence(forecast_hazard=1, obs_antecedent=1, temporal_persist=0, spatial_coverage=1, data_confidence=1),
    RiskEvidence(forecast_hazard=1, obs_antecedent=1, temporal_persist=1, spatial_coverage=1, data_confidence=1),
    RiskEvidence(forecast_hazard=2, obs_antecedent=1, temporal_persist=1, spatial_coverage=2, data_confidence=1),
    RiskEvidence(forecast_hazard=2, obs_antecedent=2, temporal_persist=1, spatial_coverage=2, data_confidence=1),
    RiskEvidence(forecast_hazard=2, obs_antecedent=2, temporal_persist=1, spatial_coverage=2, data_confidence=0),
]

with Timer('DBN sequence inference (7 days)'):
    seq_results = model.infer_sequence(sequence, initial_api_state=0)

# Extract time-series of P(High risk)
days = list(range(1, len(sequence) + 1))
p_high   = [r.get('probabilities', [0, 0, 0])[2] for r in seq_results]
p_medium = [r.get('probabilities', [0, 0, 0])[1] for r in seq_results]
p_low    = [r.get('probabilities', [0, 0, 0])[0] for r in seq_results]

fig, ax = plt.subplots(figsize=(10, 4))
ax.stackplot(days, p_low, p_medium, p_high,
             labels=['Low', 'Medium', 'High'],
             colors=['#2ecc71', '#f39c12', '#e74c3c'], alpha=0.85)
ax.set_xlim(1, len(days))
ax.set_ylim(0, 1)
ax.set_xlabel('Day')
ax.set_ylabel('Probability')
ax.set_title('CRMA DBN — Flood Risk Evolution Over 7-Day Sequence')
ax.legend(loc='upper left')
ax.axvline(x=5, color='gray', linestyle='--', alpha=0.5, label='Event onset')
plt.tight_layout()
plt.show()

METRICS['C3_max_p_high'] = max(p_high)
rprint(f'Peak P(High risk): [bold red]{max(p_high):.3f}[/] on day {p_high.index(max(p_high)) + 1}')

### C3.4 Risk sensitivity analysis — API state effect

In [ ]:
# How much does antecedent soil moisture (API) amplify risk?
high_hazard_ev = RiskEvidence(
    forecast_hazard=2, obs_antecedent=2, temporal_persist=1,
    spatial_coverage=2, data_confidence=1
)

api_states = [0, 1, 2]  # Dry, Normal, Saturated
api_labels = ['Dry', 'Normal', 'Saturated']
api_risk_high = []

for api in api_states:
    res = model.infer(high_hazard_ev, api_state=api)
    api_risk_high.append(res.get('probabilities', [0, 0, 0])[2])

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

bars = ax1.bar(api_labels, api_risk_high, color=['#3498db', '#f39c12', '#e74c3c'])
ax1.set_ylim(0, 1)
ax1.set_ylabel('P(High Risk)')
ax1.set_title('API State Effect on P(High Risk)\n(extreme hazard scenario)')
for bar, v in zip(bars, api_risk_high):
    ax1.text(bar.get_x() + bar.get_width()/2, v + 0.02, f'{v:.3f}', ha='center', fontweight='bold')

# Risk breakdown for all scenarios × API states
scenario_names = ['Low', 'Moderate', 'High', 'High+Sat']
scenario_evs   = list(scenarios.values())
heatmap_data   = np.zeros((len(scenario_evs), len(api_states)))
for i, ev in enumerate(scenario_evs):
    for j, api in enumerate(api_states):
        res = model.infer(ev, api_state=api)
        heatmap_data[i, j] = res.get('probabilities', [0, 0, 0])[2]

sns.heatmap(heatmap_data, annot=True, fmt='.3f', cmap='RdYlGn_r',
            xticklabels=api_labels, yticklabels=scenario_names,
            vmin=0, vmax=1, ax=ax2)
ax2.set_title('P(High Risk) Heatmap\nScenario × API State')
ax2.set_xlabel('Antecedent Precipitation Index (API)')

plt.tight_layout()
plt.show()

---
## Final — Pipeline Metrics Dashboard

In [ ]:
fig = plt.figure(figsize=(16, 8))
fig.patch.set_facecolor('#1a1a2e')
gs  = gridspec.GridSpec(2, 4, figure=fig, hspace=0.5, wspace=0.4)

def metric_box(ax, value, label, color='#00d4aa', fmt=None):
    ax.set_facecolor('#16213e')
    ax.set_xticks([]); ax.set_yticks([])
    for sp in ax.spines.values():
        sp.set_edgecolor(color); sp.set_linewidth(2)
    text = fmt.format(value) if fmt else str(value)
    ax.text(0.5, 0.55, text,  ha='center', va='center', fontsize=22, fontweight='bold', color=color, transform=ax.transAxes)
    ax.text(0.5, 0.18, label, ha='center', va='center', fontsize=9,  color='#aaaaaa',  transform=ax.transAxes)

metric_box(fig.add_subplot(gs[0, 0]), METRICS.get('C1_catalog_rows', 0),   'Catalog files',        '#00d4aa', '{:,}')
metric_box(fig.add_subplot(gs[0, 1]), METRICS.get('C1_dates_covered', 0),  'Dates covered',        '#00d4aa', '{:,}')
metric_box(fig.add_subplot(gs[0, 2]), METRICS.get('C1_pct_coverage', 0),   'Coverage Q1–Q2 2024',  '#f0c040', '{:.1f}%')
metric_box(fig.add_subplot(gs[0, 3]), METRICS.get('C1_catalog_gb', 0),     'Catalog size (GB)',    '#5db8fe', '{:.1f}')
metric_box(fig.add_subplot(gs[1, 0]), METRICS.get('C2_exceedance_maps', 0),'Exceedance maps',      '#ff6b6b', '{}')
metric_box(fig.add_subplot(gs[1, 1]), len(METRICS.get('C2_windows', [])),  'Accum. windows',       '#ff6b6b', '{}')
metric_box(fig.add_subplot(gs[1, 2]), METRICS.get('C3_max_p_high', 0),     'Peak P(High risk)',    '#c084fc', '{:.3f}')
metric_box(fig.add_subplot(gs[1, 3]), METRICS.get('C3_scenarios', 0),      'CRMA scenarios tested','#c084fc', '{}')

fig.suptitle('GIK-IceChain v2.0 — Pipeline Metrics', fontsize=16, fontweight='bold', color='white', y=1.01)
plt.savefig('/tmp/gik_metrics_dashboard.png', dpi=150, bbox_inches='tight', facecolor='#1a1a2e')
plt.show()
rprint('[bold green]✓ Dashboard saved to /tmp/gik_metrics_dashboard.png[/]')

In [ ]:
# Final summary table
t = Table(title='GIK-IceChain v2.0 — End-to-End Run Summary', show_header=True, header_style='bold white on blue')
t.add_column('Stage', style='bold')
t.add_column('Component')
t.add_column('Result', justify='right')
t.add_column('Status', justify='center')

t.add_row('C1', 'HF Catalog load',          f"{METRICS.get('C1_catalog_rows',0):,} files",           '[green]✓[/]')
t.add_row('C1', 'Parquet paths resolved',   f"{METRICS.get('C1_paths_resolved',0)} files (1 date)",  '[green]✓[/]')
t.add_row('C1', 'VirtualiZarr 2.x',         f"{METRICS.get('C1_virtual_vars',0)} virtual vars",      '[green]✓[/]')
t.add_row('C1', 'Coverage gap analysis',    f"{METRICS.get('C1_pct_coverage',0):.1f}% covered",      '[green]✓[/]')
t.add_row('C2', 'Rolling accumulations',    f"{len(METRICS.get('C2_windows',[]))} windows",          '[green]✓[/]')
t.add_row('C2', 'Exceedance probabilities', f"{METRICS.get('C2_exceedance_maps',0)} maps",           '[green]✓[/]')
t.add_row('C3', 'CRMA model build',         'DiscreteBN + DBN',                                       '[green]✓[/]')
t.add_row('C3', 'Single-step inference',    f"{METRICS.get('C3_scenarios',0)} scenarios",            '[green]✓[/]')
t.add_row('C3', 'DBN sequence inference',   f"Peak P(High)={METRICS.get('C3_max_p_high',0):.3f}",   '[green]✓[/]')
console.print(t)

---
*GIK-IceChain v2.0 — ECMWF Code for Earth 2026, Challenge 41. Zero-cost cloud-native flood risk pipeline for East Africa.*